# News2Stock Analyser 학습 데이터 준비

News2Stock Analyser는 경제 뉴스 원문을 입력받아 관련 상장 종목과 긍정·부정 근거를 JSON으로 생성하도록 학습하는 예제이다. 이 노트북에서는 공개 뉴스 데이터에서 경제 기사 5건을 선택하고, 관리형 LLM으로 합성 정답을 만든 뒤 `system`·`user`·`assistant` 형식의 SFT 데이터로 저장한다.

> **실행 환경:** 이 데이터 준비 실습은 GPU가 필요하지 않으므로 로컬 PyCharm Notebook에서 진행한다. 인터넷 연결과 로컬 `.env`의 `OPENAI_API_KEY`, Hub 업로드 단계의 쓰기 권한 `HF_TOKEN`이 필요하다.

이 노트북의 5건 Dataset과 개인 Hub 업로드는 schema와 업로드 과정을 확인하는 실습이다. 다음 04·05번 LoRA·QLoRA 실습은 이 5건을 사용하지 않고, 완성된 1,000건 공개 Dataset을 별도로 내려받아 학습 800건·평가 200건으로 나누어 사용한다.


## 실습 패키지 설치

`datasets`는 Hugging Face 데이터셋을 읽고 만들며,  `openai`는 strict JSON Schema를 적용한 합성 정답 생성에 사용한다.

이 노트북은 로컬에서 실행하며, 내려받은 Hugging Face 파일은 별도 설정이 없으면 사용자 계정의 기본 cache에 저장된다.



In [1]:
import sys

%pip install datasets==3.6.0 openai==2.15.0 python-dotenv pandas


   ---------------------------------------- 0.0/1.1 MB ? eta -:--:--
   ---------------------------------------- 1.1/1.1 MB 7.4 MB/s  0:00:00

  Attempting uninstall: fsspec

    Found existing installation: fsspec 2026.7.0

    Uninstalling fsspec-2026.7.0:

      Successfully uninstalled fsspec-2026.7.0

   ---------------------------------------- 0/5 [fsspec]
   ---------------------------------------- 0/5 [fsspec]
   ---------------------------------------- 0/5 [fsspec]
   ---------------------------------------- 0/5 [fsspec]
   ---------------------------------------- 0/5 [fsspec]
   ---------------------------------------- 0/5 [fsspec]
   ---------------------------------------- 0/5 [fsspec]
   ---------------------------------------- 0/5 [fsspec]
   ---------------------------------------- 0/5 [fsspec]
   ---------------------------------------- 0/5 [fsspec]
   ---------------------------------------- 0/5 [fsspec]
   ---------------------------------------- 0/5 [fsspec]
   -----

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-openai 1.6.0 requires openai<4.0.0,>=2.45.0, but you have openai 2.15.0 which is incompatible.

[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: C:\Users\playdata2\miniforge3\envs\llm_env\python.exe -m pip install --upgrade pip


## 원본 뉴스 데이터셋 로드

Hugging Face의 `DatasetDict`는 train·validation·test split을 함께 보관한다. 이 실습은 합성 학습 데이터를 만들기 위해 train split을 사용한다.

데이터 출처: [daekeun-ml/naver-news-summarization-ko](https://huggingface.co/datasets/daekeun-ml/naver-news-summarization-ko)

공개 Dataset이므로 내려받을 때 `HF_TOKEN`은 필요하지 않다. 최초 실행에는 인터넷 연결과 cache 공간이 필요하며, 이후에는 `HF_HOME`의 cache를 재사용한다.


In [2]:
from datasets import load_dataset

DATASET_ID = "daekeun-ml/naver-news-summarization-ko"

dataset = load_dataset(DATASET_ID)

print(dataset)

README.md:   0%|          | 0.00/4.69k [00:00<?, ?B/s]

C:\Users\playdata2\miniforge3\envs\llm_env\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\playdata2\.cache\huggingface\hub\datasets--daekeun-ml--naver-news-summarization-ko. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


train.csv: reconstructing file:   0%|          |  0.00B / 66.3MB            

train.csv: downloading bytes:           |  0.00B            

validation.csv:   0%|          | 0.00/7.45M [00:00<?, ?B/s]

test.csv:   0%|          | 0.00/8.17M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/22194 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2466 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2740 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['date', 'category', 'press', 'title', 'document', 'link', 'summary'],
        num_rows: 22194
    })
    validation: Dataset({
        features: ['date', 'category', 'press', 'title', 'document', 'link', 'summary'],
        num_rows: 2466
    })
    test: Dataset({
        features: ['date', 'category', 'press', 'title', 'document', 'link', 'summary'],
        num_rows: 2740
    })
})


## train split에서 경제 뉴스 선택

원본 데이터 구성처럼 train split에서 `category == "economy"`인 행만 선택한다. 제목과 본문은 앞뒤 공백을 제거하고, 두 값이 모두 있는 기사만 합성 정답 생성 후보로 남긴다.


In [7]:
import pandas as pd

# 경제 카테고리만 필터링(17,088)
economy_dataset = dataset["train"].filter(
    lambda row: row['category'] == 'economy'
)

economy_df = economy_dataset.to_pandas()

# string 변환 + strip() + 제목, 본문만 남기기
for col in ['title', 'document']:
    economy_df[col] = economy_df[col].astype("string").str.strip()

# 빈 문자열을 결측으로 통일한 뒤,
# title, document가 모두 있는 행만 0 번 부터 번호 매기기
economy_df = (
    economy_df.replace({
        'title': {"": pd.NA},
        'document': {"": pd.NA}
    }).dropna(subset=['title', 'document'])
    .reset_index(drop=True)
)

display(economy_df) # 결측치 제거 -> 17,088


,date,category,press,title,document,link,summary
0,2022-07-03 17:14:37,economy,YTN,추경호 중기 수출지원 총력 무역금융 40조 확대,앵커 정부가 올해 하반기 우리 경제의 버팀목인 수출 확대를 위해 총력을 기울이기로 ...,https://n.news.naver.com/mnews/article/052/000...,"올해 상반기 우리나라 무역수지는 역대 최악인 103억 달러 적자를 기록한 가운데, ..."
1,2022-07-04 08:07:12,economy,아시아경제,해산물·주류 무제한 인터컨티넨탈 뷔페 여름 한정 페스타 연다,문어 랍스터 대게 갑오징어 새우 소라 등 해산물 활용 미국식 해물찜 시푸드 보일 준...,https://n.news.naver.com/mnews/article/277/000...,인터엑스 1층 뷔페 레스토랑 브래서리는 오는 6일부터 8월31일까지 쿨 섬머 페스타...
2,2022-07-01 08:51:12,economy,뉴시스,에디슨이노 이승훈 제이스페이스 대표 사내이사 선임,기사내용 요약 우주발사체 사업 본격화 서울 뉴시스 김경택 기자 에디슨이노가 우주발사...,https://n.news.naver.com/mnews/article/003/001...,에디슨이노는 1일 임시주주총회를 통해 사명을 이노시스 로 변경하고 이승훈 제이스페이...
3,2022-07-01 16:11:01,economy,머니투데이,SK바사 해외 사업 조직 개편… 글로벌 탑티어 기업으로 성장 박차,SK바이오사이언스가 글로벌 사업의 고도화를 위해 조직 개편을 단행했다. SK바이오사...,https://n.news.naver.com/mnews/article/008/000...,SK바이오사이언스가 글로벌 사업의 고도화를 위해 기존 해외사업개발실을 백신사업뿐만 ...
4,2022-07-01 21:48:04,economy,경향신문,금융당국 “증시 변동성 완화 조치”,4일부터 석달간 증권사 신용융자담보비율 유지의무 면제 금융당국이 코스피지수가 장중 ...,https://n.news.naver.com/mnews/article/032/000...,1일 1일 금융위원회는 증권 유관기관과 금융시장합동점검회의를 열고 코스피지수가 장중...
...,...,...,...,...,...,...,...
17083,2022-07-04 09:28:04,economy,한국경제TV,수도권 아파트 청약경쟁률 작년 30대 1→올해 13대 1,올해 상반기 수도권을 중심으로 아파트 청약 시장이 침체국면이다. 4일 리얼투데이 조...,https://n.news.naver.com/mnews/article/215/000...,4일 리얼투데이 조사에 따르면 올해 상반기 전국 아파트 공공·민간 사전청약 아파트는...
17084,2022-07-01 12:10:59,economy,뉴스1,가격 인하된 수입 돼지고기 할당관세 0%,세종 뉴스1 김기남 기자 정부가 돼지고기 가격 안정을 위해 수입 신고되는 냉장 냉동...,https://n.news.naver.com/mnews/article/421/000...,1일 1일 오전 세종시 이마트 세종점에서 고객들이 가격이 대폭 인하된 캐나다산 돼지...
17085,2022-07-04 15:26:03,economy,서울경제,“탄소중립 앞장” 대우조선 ESG위원회 신설,보고서 발간 등 지속가능경영 힘써 대우조선해양 통합보고서. 사진제공 대우조선해양 서...,https://n.news.naver.com/mnews/article/011/000...,4조선해양이 환경·사회·지배구조 ESG 경영 확대를 위해 ESG경영 추진을 위한 각...
17086,2022-07-01 16:34:03,economy,이데일리,신원종합개발 단기 차입금 증가 결정,이데일리 김소연 기자 신원종합개발 017000 은 단기 차입금 합계가 198억800...,https://n.news.naver.com/mnews/article/018/000...,1일종합개발은 신원종합개발 017000 은 단기 차입금 합계가 자기자본 대비 17....


### 경제 뉴스 표본 확인

합성 정답을 만들기 전에 제목과 본문이 올바르게 선택되었는지 앞의 두 행을 확인한다. 전체 본문이 길 수 있으므로 화면에서는 필요한 열만 표시한다.


In [9]:
display(economy_df[['title', 'document']].head(2))

,title,document
0,추경호 중기 수출지원 총력 무역금융 40조 확대,앵커 정부가 올해 하반기 우리 경제의 버팀목인 수출 확대를 위해 총력을 기울이기로 ...
1,해산물·주류 무제한 인터컨티넨탈 뷔페 여름 한정 페스타 연다,문어 랍스터 대게 갑오징어 새우 소라 등 해산물 활용 미국식 해물찜 시푸드 보일 준...


## system 지시문과 strict JSON Schema 정의

`system`은 모델의 금융 뉴스 분석 역할을 고정하고, `user` 템플릿은 기사 제목과 본문을 전달한다. `NEWS_ANALYSIS_SCHEMA`는 Responses API가 반환할 key와 자료형을 제한하며, 모든 레코드가 같은 구조를 갖도록 여덟 필드를 필수로 둔다.


In [10]:
SYSTEM_MESSAGE = (
    "금융·경제 뉴스의 사실에 근거해 상장 종목 영향을 분석한다. "
    "불확실한 종목을 만들지 말고 교육용 구조화 데이터만 반환한다."
)

USER_MESSAGE_TEMPLATE = """뉴스 제목과 본문을 읽고 상장 종목 영향을 분석한다.
연관 종목이 없으면 stock_related를 false로 두고 영향 필드는 빈 값으로 반환한다.
추측한 종목이나 투자 권유를 만들지 않는다.

뉴스:
{news}
"""

# 문자열 목록 schema(구조)
# - 종목명과 근거 키워드의 공통 자료형으로 재사용 예정
string_list_schema = {
    "type": "array",
    "items": {"type": "string", "minLength": 1},
}

# LLM이 답변하길 원하는 JSON Schema의 필드 자료형 정의
NEWS_ANALYSIS_FIELDS = {
    "stock_related": {"type": "boolean"},
    "positive_stocks": string_list_schema,
    "positive_reasons": {"type": "string"},
    "positive_keywords": string_list_schema,
    "negative_stocks": string_list_schema,
    "negative_reasons": {"type": "string"},
    "negative_keywords": string_list_schema,
    "summary": {"type": "string", "minLength": 1},
}

# 전체 JSON Schema 정의
NEWS_ANALYSIS_SCHEMA = {
    "type": "object",
    "properties": NEWS_ANALYSIS_FIELDS,
    "required": list(NEWS_ANALYSIS_FIELDS),
    "additionalProperties": False,
}

# LLM에 적용할 설정 dict
# - "strict": True : 설정 내용(schema)를 엄격하게 지켜라
# - "schema": NEWS_ANALYSIS_SCHEMA == JSON Schema
RESPONSE_TEXT_CONFIG = {
    "format": {
        "type": "json_schema",
        "name": "news_stock_analysis",
        "strict": True,
        "schema": NEWS_ANALYSIS_SCHEMA,
    }
}


## OpenAI 합성 정답 생성 전 확인

다음 셀은 `OPENAI_API_KEY`를 읽어 Client만 준비하므로 비용이 발생하지 않는다. 실제 뉴스 전송과 비용은 5건의 합성 정답을 생성하는 호출 셀을 실행할 때 발생한다. 로컬 프로젝트의 `.env`에 `OPENAI_API_KEY`를 준비한다. 기본 모델은 `gpt-5.6-luna`이며, `OPENAI_CHAT_MODEL`은 다른 허용 모델로 바꿀 때만 지정한다.

`.env`를 추가하거나 수정했다면 로컬 Notebook kernel을 재시작한 뒤 처음부터 다시 실행한다. API key 값은 출력하거나 노트북에 직접 작성하지 않는다.

`gpt-5.6-luna`는 Chat Completions도 지원하지만 이 실습은 Responses API를 사용한다. Responses API에서는 대화형 `messages` 대신 `instructions`와 `input`으로 지시와 입력을 나누고, 생성 상한은 `max_output_tokens`, 최종 문자열은 `response.output_text`로 다룬다.


In [12]:
import os

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(override=False)

OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]
OPENAI_CHAT_MODEL = os.getenv(
    "OPENAI_CHAT_MODEL",
    "gpt-5.6-luna",
)
GENERATION_LIMIT = 5

# OpenAI API 요청용 Client 객체 생성
client = OpenAI(
    api_key=OPENAI_API_KEY,
    timeout=60,
    max_retries=0
)

# LLM을 이용한 합성 정답 생성 대상
generation_df = economy_df.head(GENERATION_LIMIT).copy()
generation_df["user"] = (
    generation_df["title"] + "\n" + generation_df["document"]
)
news_inputs = generation_df["user"].tolist()

print("합성 정답 생성 대상:", len(news_inputs))
display(news_inputs)

합성 정답 생성 대상: 5


['추경호 중기 수출지원 총력 무역금융 40조 확대\n앵커 정부가 올해 하반기 우리 경제의 버팀목인 수출 확대를 위해 총력을 기울이기로 했습니다. 특히 수출 중소기업의 물류난 해소를 위해 무역금융 규모를 40조 원 이상 확대하고 물류비 지원과 임시선박 투입 등을 추진하기로 했습니다. 류환홍 기자가 보도합니다. 기자 수출은 최고의 실적을 보였지만 수입액이 급증하면서 올해 상반기 우리나라 무역수지는 역대 최악인 103억 달러 적자를 기록했습니다. 정부가 수출확대에 총력을 기울이기로 한 것은 원자재 가격 상승 등 대외 리스크가 가중되는 상황에서 수출 증가세 지속이야말로 한국경제의 회복을 위한 열쇠라고 본 것입니다. 추경호 경제부총리 겸 기획재정부 장관 정부는 우리 경제의 성장엔진인 수출이 높은 증가세를 지속할 수 있도록 총력을 다하겠습니다. 우선 물류 부담 증가 원자재 가격 상승 등 가중되고 있는 대외 리스크에 대해 적극 대응하겠습니다. 특히 중소기업과 중견기업 수출 지원을 위해 무역금융 규모를 연초 목표보다 40조 원 늘린 301조 원까지 확대하고 물류비 부담을 줄이기 위한 대책도 마련했습니다. 이창양 산업통상자원부 장관 국제 해상운임이 안정될 때까지 월 4척 이상의 임시선박을 지속 투입하는 한편 중소기업 전용 선복 적재 용량 도 현재보다 주당 50TEU 늘려 공급하겠습니다. 하반기에 우리 기업들의 수출 기회를 늘리기 위해 2 500여 개 수출기업을 대상으로 해외 전시회 참가를 지원하는 등 마케팅 지원도 벌이기로 했습니다. 정부는 또 이달 중으로 반도체를 비롯한 첨단 산업 육성 전략을 마련해 수출 증가세를 뒷받침하고 에너지 소비를 줄이기 위한 효율화 방안을 마련해 무역수지 개선에 나서기로 했습니다. YTN 류환홍입니다.',
 '해산물·주류 무제한 인터컨티넨탈 뷔페 여름 한정 페스타 연다\n문어 랍스터 대게 갑오징어 새우 소라 등 해산물 활용 미국식 해물찜 시푸드 보일 준비 7 8월 2만5000원 추가 시 와인 5종 및 생맥주 무제한 제공 인터컨티넨탈 서울 코엑스 

### 기사 한 건을 한 줄 JSON으로 변환

`client.responses.create()`는 `instructions`에 공통 역할, `input`에 현재 뉴스, `text`에 strict JSON schema를 전달한다. `reasoning.effort='low'`는 합성 정답 생성에 필요한 추론은 유지하면서 응답 시간과 token 사용을 줄이는 설정이다. `max_output_tokens`에는 화면에 보이는 답변뿐 아니라 reasoning token도 포함되므로 2,000으로 늘려 긴 JSON이 중간에서 잘리지 않게 한다. 응답 상태가 `completed`가 아니면 JSON 파싱 전에 중단 원인을 표시하고, 완성된 `response.output_text`만 `json.loads()`로 파싱한다.


In [13]:
import json

def analyze_news(news: str) -> str:

    response = client.responses.create(
        model=OPENAI_CHAT_MODEL, # 모델명
        instructions=SYSTEM_MESSAGE, # 지침
        input=USER_MESSAGE_TEMPLATE.format(news=news), # 사용자 입력,
        reasoning={"effort": "low"}, # 추론 레벨
        text=RESPONSE_TEXT_CONFIG, # LLM 답변(출력) 설정
        max_output_tokens=2000 # 응답 토큰 수 제한
    )

    # token 상한 등으로 응답이 중간 종료되면 불완전한 문자열을 JSON으로 파싱하지 않는다.
    if response.status != "completed":
        reason = getattr(response.incomplete_details, "reason", None)
        raise RuntimeError(
            f"응답이 완성되지 않았다: status={response.status}, reason={reason}"
        )

    payload = json.loads(response.output_text)

    # ensure_ascii=False는 한글, sort_keys는 key 순서, separators는 한 줄 JSON 형식을 고정한다.
    return json.dumps(
        payload,
        ensure_ascii=False,
        sort_keys=True,
        separators=(",", ":"),
    )

### 5건의 assistant 응답 생성

앞에서 준비한 뉴스 5건을 순서대로 호출한다. `results`의 같은 index에는 `news_inputs` 기사에 대한 한 줄 JSON 문자열이 저장된다. 호출량을 5건으로 제한해 실습 시간과 API 비용을 줄이고, 데이터 구조를 확인하는 데 집중한다.


In [14]:
results = []

# news_inputs 한 항목마다 API를 한 번 호출하고 같은 순서로 assistant 문자열을 누적한다.
for news in news_inputs:
    results.append(analyze_news(news))

print("생성 완료:", len(results))
print(results[0])

생성 완료: 5
{"negative_keywords":[],"negative_reasons":"","negative_stocks":[],"positive_keywords":[],"positive_reasons":"","positive_stocks":[],"stock_related":false,"summary":"정부가 수출 중소·중견기업을 대상으로 무역금융을 301조 원까지 확대하고 물류비 지원, 임시선박 투입, 해외 전시회 참가 지원 등을 추진한다는 내용이다. 반도체 등 첨단산업 육성 방침도 언급됐지만 구체적인 상장 종목이나 기업이 제시되지 않아 종목별 영향은 판단하기 어렵다."}


## JSON 파싱과 SFT 레코드 구성

모든 assistant 문자열을 다시 파싱해 유효한 JSON인지 확인한다. 이후 각 뉴스에 같은 system 지시문과 대응하는 assistant object를 붙여 `system`·`user`·`assistant` 세 열을 만든다. 04·05번 노트북은 이 object를 `json.dumps()`로 한 번만 직렬화한다.


In [15]:

# json.loads(result):
# - s는 str을 의미
# - result(str)를 역질렬화해서 dict 형태로 변환하는 함수
# - 만약 dict로 바꿀 수 없는 양식이면 Error 발생
parsed_results = [json.loads(result) for result in results]

result_df = generation_df.copy()
result_df["system"] = SYSTEM_MESSAGE
result_df["assistant"] = parsed_results
result_df = result_df[["system", "user", "assistant"]]

print("JSON 파싱 완료:", len(parsed_results))
display(result_df.head())

JSON 파싱 완료: 5


,system,user,assistant
0,금융·경제 뉴스의 사실에 근거해 상장 종목 영향을 분석한다. 불확실한 종목을 만들지...,추경호 중기 수출지원 총력 무역금융 40조 확대\n앵커 정부가 올해 하반기 우리 경...,"{'negative_keywords': [], 'negative_reasons': ..."
1,금융·경제 뉴스의 사실에 근거해 상장 종목 영향을 분석한다. 불확실한 종목을 만들지...,해산물·주류 무제한 인터컨티넨탈 뷔페 여름 한정 페스타 연다\n문어 랍스터 대게 갑...,"{'negative_keywords': [], 'negative_reasons': ..."
2,금융·경제 뉴스의 사실에 근거해 상장 종목 영향을 분석한다. 불확실한 종목을 만들지...,에디슨이노 이승훈 제이스페이스 대표 사내이사 선임\n기사내용 요약 우주발사체 사업 ...,"{'negative_keywords': [], 'negative_reasons': ..."
3,금융·경제 뉴스의 사실에 근거해 상장 종목 영향을 분석한다. 불확실한 종목을 만들지...,SK바사 해외 사업 조직 개편… 글로벌 탑티어 기업으로 성장 박차\nSK바이오사이언...,"{'negative_keywords': [], 'negative_reasons': ..."
4,금융·경제 뉴스의 사실에 근거해 상장 종목 영향을 분석한다. 불확실한 종목을 만들지...,금융당국 “증시 변동성 완화 조치”\n4일부터 석달간 증권사 신용융자담보비율 유지의...,"{'negative_keywords': [], 'negative_reasons': ..."


## `train.json` 저장과 재확인

학습 데이터는 로컬 프로젝트의 `data/news2stock/train.json`에 JSON 배열로 저장한다. 이 파일은 생성 결과를 눈으로 확인하고 다음 Hub 업로드 셀에서 Hugging Face Dataset으로 변환하기 위한 중간 파일이다. 저장 직후 같은 파일을 다시 읽어 레코드 수와 세 필드가 유지되는지 확인한다.


In [16]:
from pathlib import Path

OUTPUT_PATH = Path("data/news2stock/train.json")

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

# orient=records: 행별 object 배열
# force_ascii=False: 한글
# indent=2: 읽기 쉬운 들여쓰기 설정
result_df.to_json(
    OUTPUT_PATH,
    orient="records",
    force_ascii=False,
    indent=2,
)

# 저장 직후 다시 읽어 파일의 레코드 수와 key 계약을 확인한다.
with OUTPUT_PATH.open("r", encoding="utf-8") as file:
    saved_records = json.load(file)

print("저장 경로:", OUTPUT_PATH.as_posix())
print("저장 레코드:", len(saved_records))
print("첫 레코드 key:", list(saved_records[0]))

저장 경로: data/news2stock/train.json
저장 레코드: 5
첫 레코드 key: ['system', 'user', 'assistant']


## 개인 Hugging Face Hub 저장소에 업로드

앞에서 저장한 5건의 `train.json`을 Hugging Face Dataset으로 변환하고 개인 Hub 저장소에 업로드한다. `train.json`은 03번 노트북 안에서 결과를 확인하고 업로드하기 위한 로컬 중간 파일이다. 이 업로드는 Dataset 저장소를 직접 만들어 보는 연습이며, 04번 파인튜닝의 입력으로 사용하지 않는다.

로컬 `.env`의 `HF_TOKEN`에는 자신의 저장소에 쓸 수 있는 권한이 필요하고, 코드의 `HF_DATASET_REPO_ID`에는 `사용자명/저장소명` 형식의 목적지를 설정한다. `HF_TOKEN`은 앞의 공개 Dataset 다운로드에는 필요하지 않고 이 업로드 셀에서 처음 사용한다.

04번은 완성된 1,000건 공개 저장소 `shqkel/naver-economy-news2stock`을 별도로 내려받아 학습 800건·평가 200건으로 분할한다. 따라서 개인 저장소의 5건은 Hub 업로드 성공과 `system`·`user`·`assistant` 세 열만 확인하면 된다.


In [17]:
from datasets import Dataset

load_dotenv(override=False)
HF_TOKEN = os.environ["HF_TOKEN"]
HF_DATASET_REPO_ID = 'goat-baek/naver-economy-news2stock'

# pathlib.as_posix() : 경로 구분자를 '/'로 통일
# json 읽어오기
hub_dataset = Dataset.from_json(OUTPUT_PATH.as_posix())

push_result = hub_dataset.push_to_hub(
    repo_id=HF_DATASET_REPO_ID,
    token=HF_TOKEN,
)
print(push_result)


Generating train split: 0 examples [00:00, ? examples/s]

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

https://huggingface.co/datasets/goat-baek/naver-economy-news2stock/commit/e7e76927bbb300ca38be01faab923c02fa8a5994
